# Notebook 04 — LSTM & GRU Models

**Goal:** Train sequential deep learning models (LSTM, GRU) on LOB sequence data.

These models can capture temporal patterns that classical ML misses — the *order* of LOB events matters.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.preprocessing import StandardScaler

from src.utils import set_seed, evaluate_classifier, plot_confusion_matrix
from src.models import LSTMClassifier, GRUClassifier, LOBDataset
from src.data_loader import create_sequences, train_val_test_split

set_seed(42)
pl.seed_everything(42)
%matplotlib inline

RESULTS = Path('../results')
SEQ_LEN = 50       # lookback window: 50 timesteps
BATCH_SIZE = 256
MAX_EPOCHS = 15
print(f'Device: cpu | Seq len: {SEQ_LEN} | Batch: {BATCH_SIZE}')

## 1. Load & Prepare Sequence Data

In [ ]:
df = pd.read_parquet('../data/processed/features.parquet')
feature_cols = [c for c in df.columns if not c.startswith('label')]

X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int64)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Standardize
scaler = StandardScaler()
n_train = int(len(X) * 0.7)
scaler.fit(X[:n_train])
X_scaled = scaler.transform(X)

# Create sequences
X_seq, y_seq = create_sequences(X_scaled, y, seq_len=SEQ_LEN)

# Chronological split
splits = train_val_test_split(X_seq, y_seq, train_ratio=0.7, val_ratio=0.15)

train_ds = LOBDataset(splits['X_train'], splits['y_train'])
val_ds = LOBDataset(splits['X_val'], splits['y_val'])
test_ds = LOBDataset(splits['X_test'], splits['y_test'])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

n_features = X_seq.shape[2]
print(f'n_features: {n_features}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

## 2. Train LSTM

In [ ]:
lstm_model = LSTMClassifier(
    n_features=n_features, hidden_dim=64,
    num_layers=2, n_classes=3, dropout=0.3, lr=1e-3
)

trainer_lstm = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator='cpu',
    enable_progress_bar=True, enable_model_summary=True,
    callbacks=[pl.callbacks.EarlyStopping('val_loss', patience=5)],
    logger=False,
)

trainer_lstm.fit(lstm_model, train_dl, val_dl)
print('LSTM training complete.')

In [ ]:
# Evaluate LSTM
lstm_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_batch, y_batch in test_dl:
        logits = lstm_model(x_batch)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.numpy())
        all_labels.append(y_batch.numpy())

lstm_preds = np.concatenate(all_preds)
lstm_labels = np.concatenate(all_labels)

lstm_metrics = evaluate_classifier(lstm_labels, lstm_preds, title='LSTM')
plot_confusion_matrix(lstm_labels, lstm_preds, title='LSTM',
                     save_path=str(RESULTS / 'plots' / 'cm_lstm.png'))

torch.save(lstm_model.state_dict(), RESULTS / 'models' / 'lstm.pt')

## 3. Train GRU

In [ ]:
gru_model = GRUClassifier(
    n_features=n_features, hidden_dim=64,
    num_layers=2, n_classes=3, dropout=0.3, lr=1e-3
)

trainer_gru = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator='cpu',
    enable_progress_bar=True, enable_model_summary=True,
    callbacks=[pl.callbacks.EarlyStopping('val_loss', patience=5)],
    logger=False,
)

trainer_gru.fit(gru_model, train_dl, val_dl)
print('GRU training complete.')

In [ ]:
# Evaluate GRU
gru_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_batch, y_batch in test_dl:
        logits = gru_model(x_batch)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.numpy())
        all_labels.append(y_batch.numpy())

gru_preds = np.concatenate(all_preds)
gru_labels = np.concatenate(all_labels)

gru_metrics = evaluate_classifier(gru_labels, gru_preds, title='GRU')
plot_confusion_matrix(gru_labels, gru_preds, title='GRU',
                     save_path=str(RESULTS / 'plots' / 'cm_gru.png'))

torch.save(gru_model.state_dict(), RESULTS / 'models' / 'gru.pt')

## 4. Compare LSTM vs GRU

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'LSTM', **lstm_metrics},
    {'Model': 'GRU', **gru_metrics},
])
comparison.to_csv(RESULTS / 'tables' / 'lstm_gru_results.csv', index=False)
print('\n=== LSTM vs GRU ===')
print(comparison.to_string(index=False))

# Save predictions for backtesting
np.save('../data/processed/lstm_preds.npy', lstm_preds)
np.save('../data/processed/gru_preds.npy', gru_preds)
np.save('../data/processed/test_labels.npy', lstm_labels)
print('Predictions saved for backtesting.')

## Summary

- LSTM captures temporal dependencies via forget/input/output gates
- GRU is simpler (2 gates vs 3) and often trains faster
- Both should outperform Logistic Regression; competitive with LightGBM

**Next:** Notebook 05 — Temporal Fusion Transformer